# Hồi quy Softmax (Softmax Regression)
:label:`sec_softmax`

Trong mục :numref:`sec_linear_regression`, chúng ta đã tìm hiểu về hồi quy tuyến tính (linear regression), đi qua các bước triển khai từ đầu trong :numref:`sec_linear_scratch` và sử dụng các API cấp cao của một thư viện học sâu trong :numref:`sec_linear_concise` để xử lý các tác vụ phức tạp.

Hồi quy là công cụ chúng ta tìm đến khi muốn trả lời các câu hỏi *bao nhiêu?* (ví dụ: lượng tiền hoặc số lượng). Nếu bạn muốn dự đoán giá của một ngôi nhà (bằng USD), số trận thắng của một đội bóng chày, hay số ngày một bệnh nhân phải nằm viện trước khi xuất viện, thì có lẽ bạn đang cần một mô hình hồi quy.

Tuy nhiên, ngay cả trong các mô hình hồi quy, vẫn có những điểm khác biệt quan trọng. Ví dụ, giá nhà không bao giờ âm và các thay đổi thường mang tính *tương đối* so với mức giá cơ sở. Vì thế, việc thực hiện hồi quy trên logarit của giá nhà sẽ hiệu quả hơn. Tương tự, số ngày nằm viện của bệnh nhân là một biến ngẫu nhiên *rời rạc không âm*. Do đó, phương pháp bình phương tối thiểu (least mean squares) có thể không phải là cách tiếp cận lý tưởng. Loại mô hình hóa thời gian xảy ra sự kiện này đi kèm với nhiều biến chứng khác và được giải quyết trong một lĩnh vực chuyên biệt gọi là *mô hình hóa sống sót* (survival modeling).

Điểm mấu chốt ở đây không phải là làm bạn quá tải với thông tin, mà chỉ để bạn biết rằng việc ước lượng còn nhiều điều thú vị hơn là chỉ giảm thiểu sai số bình phương. Và rộng hơn thế, học có giám sát (supervised learning) không chỉ có hồi quy. Trong phần này, chúng ta sẽ tập trung vào bài toán *phân loại* (classification), nơi chúng ta tạm gác các câu hỏi *bao nhiêu?* để tập trung vào câu hỏi *thuộc nhóm nào?*.

* Email này thuộc hộp thư rác (spam) hay hộp thư đến (inbox)?
* Khách hàng này có nhiều khả năng đăng ký hay không đăng ký dịch vụ dài hạn?
* Bức ảnh này chụp con lừa, con chó, con mèo hay con gà trống?
* Bộ phim nào mà Aston có khả năng xem tiếp theo nhất?
* Phần nào của cuốn sách mà bạn sẽ đọc tiếp theo?

Trong giao tiếp thông thường, các kỹ sư học máy thường dùng từ *phân loại* (classification) để mô tả hai bài toán có chút khác biệt:
1. **Phân loại cứng (hard assignment)**: Chỉ quan tâm đến việc xếp các mẫu dữ liệu vào hẳn một danh mục cụ thể.
2. **Phân loại mềm (soft assignment)**: Ước lượng xác suất mà mỗi danh mục có thể xảy ra.

Ranh giới giữa hai khái niệm này thường bị mờ nhạt, một phần là vì ngay cả khi chỉ cần phân loại cứng, chúng ta vẫn sử dụng các mô hình tạo ra phân loại mềm.

Ngoài ra, còn có những trường hợp một mẫu có thể có nhiều nhãn đúng cùng lúc. Ví dụ: một bài báo có thể đồng thời thuộc các chủ đề giải trí, kinh doanh và hàng không vũ trụ, chứ không chỉ một chủ đề duy nhất. Việc chỉ xếp nó vào một danh mục duy nhất sẽ không mấy hữu ích. Bài toán này được gọi là [phân loại đa nhãn (multi-label classification)](https://en.wikipedia.org/wiki/Multi-label_classification). Bạn có thể tham khảo :cite:`Tsoumakas.Katakis.2007` để có cái nhìn tổng quan và :cite:`Huang.Xu.Yu.2015` để biết thêm về một thuật toán hiệu quả khi gắn thẻ (tagging) cho hình ảnh.

## Phân loại (Classification)
:label:`subsec_classification-problem`

Để bắt đầu, chúng ta hãy xem xét một bài toán phân loại hình ảnh đơn giản. Ở đây, mỗi đầu vào là một bức ảnh xám kích thước $2\times2$. Chúng ta có thể biểu diễn giá trị của mỗi điểm ảnh bằng một số vô hướng, tạo ra 4 đặc trưng (features) $x_1, x_2, x_3, x_4$. Hơn nữa, giả sử mỗi bức ảnh chỉ thuộc về một trong ba danh mục: "mèo" (cat), "gà" (chicken), và "chó" (dog).

Tiếp theo, chúng ta phải chọn cách biểu diễn các nhãn này. Có hai cách rõ ràng:
- Cách tự nhiên nhất là chọn $y \in \{1, 2, 3\}$, trong đó các số nguyên đại diện cho $\{\textrm{chó}, \textrm{mèo}, \textrm{gà}\}$ tương ứng. Đây là một cách tuyệt vời để *lưu trữ* thông tin trong máy tính. Nếu các danh mục có thứ tự tự nhiên (ví dụ dự đoán độ tuổi: $\{\textrm{trẻ sơ sinh}, \textrm{trẻ chập chững}, \textrm{vị thành niên}, \textrm{thanh niên}, \textrm{trưởng thành}, \textrm{người già}\}$), thì việc đưa bài toán này về dạng [hồi quy thứ tự (ordinal regression)](https://en.wikipedia.org/wiki/Ordinal_regression) là hoàn toàn hợp lý.

Tuy nhiên, các bài toán phân loại thông thường không có thứ tự tự nhiên giữa các lớp. May mắn thay, các nhà thống kê đã phát minh ra một cách đơn giản để biểu diễn dữ liệu phân loại: *mã hóa một lần (one-hot encoding)*. Một vector mã hóa one-hot sẽ có số phần tử bằng đúng số lượng danh mục. Phần tử tương ứng với danh mục của mẫu dữ liệu đó sẽ được đặt bằng 1, và tất cả các phần tử khác bằng 0. Trong trường hợp của chúng ta, nhãn $y$ sẽ là một vector 3 chiều, với $(1, 0, 0)$ tương ứng với "mèo", $(0, 1, 0)$ là "gà", và $(0, 0, 1)$ là "chó":

$$y \in \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}.$$

### Mô hình tuyến tính (Linear Model)

Để ước lượng xác suất có điều kiện liên quan đến tất cả các lớp có thể xảy ra, chúng ta cần một mô hình có nhiều đầu ra (mỗi lớp một đầu ra). Để giải quyết bài toán phân loại bằng mô hình tuyến tính, chúng ta cần số lượng hàm affine bằng đúng số lượng đầu ra. Mỗi đầu ra sẽ tương ứng với một hàm affine riêng biệt. Trong ví dụ này, vì có 4 đặc trưng và 3 danh mục đầu ra, chúng ta cần 12 trọng số ($w$) và 3 hệ số chệch ($b$):

$$
\begin{aligned}
\text{# Đầu ra cho lớp thứ 1 (Mèo)} \\
o_1 &= x_1 w_{11} + x_2 w_{12} + x_3 w_{13} + x_4 w_{14} + b_1,\\
\text{# Đầu ra cho lớp thứ 2 (Gà)} \\
o_2 &= x_1 w_{21} + x_2 w_{22} + x_3 w_{23} + x_4 w_{24} + b_2,\\
\text{# Đầu ra cho lớp thứ 3 (Chó)} \\
o_3 &= x_1 w_{31} + x_2 w_{32} + x_3 w_{33} + x_4 w_{34} + b_3.
\end{aligned}
$$

Sơ đồ mạng thần kinh tương ứng được hiển thị ở hình :numref:`fig_softmaxreg`. Giống như hồi quy tuyến tính, chúng ta sử dụng một mạng thần kinh một lớp đơn duy nhất. Và vì việc tính toán mỗi đầu ra $o_1, o_2, o_3$ đều phụ thuộc vào mọi đầu vào $x_1, x_2, x_3, x_4$, nên lớp đầu ra này được gọi là *lớp kết nối đầy đủ (fully connected layer)* hoặc *lớp tuyến tính (dense layer)*.

![Hồi quy Softmax là một mạng thần kinh một lớp đơn.](http://d2l.ai/_images/softmaxreg.svg)
:label:`fig_softmaxreg`

Để viết ngắn gọn hơn, chúng ta sử dụng ký hiệu vector và ma trận:

$$\mathbf{o} = \mathbf{W} \mathbf{x} + \mathbf{b}$$

Ký hiệu này giúp việc lập trình và tính toán toán học trở nên thuận tiện hơn rất nhiều. Ở đây, tất cả trọng số được gộp vào một ma trận kích thước $3 \times 4$ và tất cả các hệ số chệch được gom vào một vector $\mathbf{b} \in \mathbb{R}^3$.

### Hàm Softmax
:label:`subsec_softmax_operation`

Nếu có một hàm mất mát (loss function) phù hợp, chúng ta có thể cố gắng trực tiếp giảm thiểu sự khác biệt giữa đầu ra $\mathbf{o}$ và nhãn $\mathbf{y}$. Mặc dù việc coi bài toán phân loại như một bài toán hồi quy vector hoạt động tốt một cách đáng ngạc nhiên, phương pháp này vẫn gặp phải các hạn chế sau:
- Không có gì đảm bảo rằng tổng các đầu ra $o_i$ sẽ bằng $1$ để có thể đại diện cho xác suất.
- Không có gì đảm bảo các đầu ra $o_i$ sẽ không âm, hoặc không vượt quá $1$.

Cả hai khía cạnh này làm cho bài toán tối ưu hóa trở nên khó giải quyết và rất nhạy cảm với các điểm dữ liệu dị biệt (outliers). Do đó, chúng ta cần một cơ chế để "nén" (squish) các đầu ra này lại.

Để đạt được mục tiêu này, chúng ta có thể sử dụng hàm mũ $P(y = i) \propto \exp o_i$. Hàm này đảm bảo rằng xác suất của một lớp tăng lên khi điểm số đầu ra $o_i$ tương ứng tăng, đồng thời giữ cho tất cả các giá trị thu được luôn không âm. Sau đó, chúng ta chuẩn hóa chúng bằng cách chia cho tổng các giá trị mũ để tổng bằng $1$. Quá trình này tạo nên hàm **softmax**:

$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{o}) \quad \text{với} \quad \hat{y}_i = \frac{\exp(o_i)}{\sum_j \exp(o_j)}.$$
:eqlabel:`eq_softmax_y_and_o`

Lưu ý rằng tọa độ lớn nhất của $\mathbf{o}$ sẽ tương ứng với lớp có khả năng xảy ra cao nhất trong $\hat{\mathbf{y}}$. Vì phép toán softmax bảo toàn thứ tự của các đối số, chúng ta không cần phải tính softmax để xác định lớp nào được gán xác suất cao nhất:

$$
\operatorname*{argmax}_j \hat y_j = \operatorname*{argmax}_j o_j.
$$

### Vector hóa (Vectorization)
:label:`subsec_softmax_vectorization`

Để cải thiện hiệu suất tính toán, chúng ta vector hóa các phép tính theo từng nhóm dữ liệu nhỏ (minibatches). Giả sử chúng ta có một minibatch $\mathbf{X} \in \mathbb{R}^{n \times d}$ gồm $n$ mẫu với số lượng đặc trưng đầu vào là $d$. Ngoài ra, giả sử chúng ta có $q$ phân lớp ở đầu ra. Khi đó, ma trận trọng số sẽ là $\mathbf{W} \in \mathbb{R}^{d \times q}$ và vector hệ số chệch là $\mathbf{b} \in \mathbb{R}^{1 \times q}$:

$$
\begin{aligned}
\mathbf{O} &= \mathbf{X} \mathbf{W} + \mathbf{b}, \\
\hat{\mathbf{Y}} &= \mathrm{softmax}(\mathbf{O}).
\end{aligned}
$$
:eqlabel:`eq_minibatch_softmax_reg`

Phép toán chủ đạo ở đây là nhân hai ma trận $\mathbf{X} \mathbf{W}$, giúp tăng tốc độ tính toán đáng kể trên phần cứng hiện đại. Hơn nữa, phép toán softmax được thực hiện trên từng hàng (rowwise): với mỗi hàng của $\mathbf{O}$, chúng ta lũy thừa tất cả các phần tử rồi chia cho tổng của chúng để chuẩn hóa.

## Hàm mất mát (Loss Function)
:label:`subsec_softmax-regression-loss-func`

Bây giờ chúng ta cần một phương pháp để tối ưu hóa độ chính xác của mô hình dự đoán. Chúng ta sẽ dựa trên phương pháp ước lượng hợp lý cực đại (Maximum Likelihood Estimation - MLE).

### Log-Likelihood

Hàm softmax cung cấp cho chúng ta một vector $\hat{\mathbf{y}}$, có thể được hiểu là xác suất có điều kiện ước lượng của mỗi lớp khi biết đầu vào $\mathbf{x}$. Giả sử nhãn thực tế $\mathbf{y}$ được biểu diễn dưới dạng mã hóa one-hot. Chúng ta có thể tối thiểu hóa log-likelihood âm (negative log-likelihood):

$$
-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}) = \sum_{i=1}^n l(\mathbf{y}^{(i)}, \hat{\mathbf{y}}^{(i)}),
$$

trong đó hàm mất mát $l$ cho cặp nhãn thực tế $\mathbf{y}$ và dự đoán $\hat{\mathbf{y}}$ là:

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$
:eqlabel:`eq_l_cross_entropy`

Hàm mất mát này thường được gọi là **hàm mất mát entropy chéo (cross-entropy loss)**. Vì $\mathbf{y}$ là một vector one-hot, tổng trên tất cả các lớp $j$ sẽ triệt tiêu chỉ còn lại một điều khoản duy nhất tương ứng với nhãn đúng.

### Softmax và Entropy chéo
:label:`subsec_softmax_and_derivatives`

Khi kết hợp hàm softmax và hàm mất mát entropy chéo, chúng ta có thu được biểu thức toán học rất đẹp:

$$
\begin{aligned}
l(\mathbf{y}, \hat{\mathbf{y}}) &= - \sum_{j=1}^q y_j \log \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} \\
&= \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j.
\end{aligned}
$$

Nếu lấy đạo hàm riêng theo bất kỳ logit $o_j$ nào, chúng ta thu được:

$$ \partial_{o_j} l(\mathbf{y}, \hat{\mathbf{y}}) = \mathrm{softmax}(\mathbf{o})_j - y_j. $$

Đạo hàm này chính là sự khác biệt giữa xác suất mà mô hình dự đoán và nhãn thực tế $y_j$. Điều này cực kỳ giống với những gì chúng ta thấy trong hồi quy tuyến tính, giúp việc tính toán gradient trong thực tế trở nên vô cùng đơn giản và trực quan.

## Cơ bản về Lý thuyết thông tin (Information Theory Basics)
:label:`subsec_info_theory_basics`

Lý thuyết thông tin giải quyết các bài toán mã hóa, giải mã, truyền tải và xử lý thông tin (dữ liệu).

### Entropy

Ý tưởng trung tâm của lý thuyết thông tin là định lượng lượng thông tin chứa trong dữ liệu. Đối với một phân phối $P$, entropy $H[P]$ của nó được định nghĩa là:

$$H[P] = \sum_j - P(j) \log P(j).$$
:eqlabel:`eq_softmax_reg_entropy`

### Surprisal (Độ bất ngờ)

Nếu chúng ta không thể dự đoán hoàn hảo mọi sự kiện, đôi khi chúng ta sẽ cảm thấy bất ngờ. Độ bất ngờ của chúng ta sẽ lớn hơn khi một sự kiện xảy ra có xác suất dự đoán thấp hơn. Claude Shannon đã sử dụng đại lượng $-\log P(j)$ để đo lường độ bất ngờ này.

### Nhìn lại Entropy chéo (Cross-Entropy)

Entropy chéo từ phân phối thực tế $P$ sang phân phối dự đoán $Q$, ký hiệu là $H(P, Q)$, là độ bất ngờ kỳ vọng của một người quan sát có niềm tin chủ quan $Q$ khi thấy dữ liệu thực tế được tạo ra theo phân phối $P$:

$$H(P, Q) \stackrel{\textrm{def}}{=} \sum_j - P(j) \log Q(j).$$

Giá trị entropy chéo nhỏ nhất đạt được khi $P=Q$.

## Tóm tắt và Thảo luận

Trong phần này, chúng ta đã tìm hiểu về hồi quy Softmax, một phương pháp tiếp cận xác suất cho phép chúng ta giải quyết bài toán phân loại trên các không gian đầu ra rời rạc. Chúng ta đã kết nối các khái niệm toán học cốt lõi từ cơ chế chuẩn hóa xác suất của Softmax đến các nguyên lý cơ bản của Lý thuyết thông tin thông qua hàm mất mát entropy chéo.

---

## Exercises

1. We can explore the connection between exponential families and softmax in some more depth.
    1. Compute the second derivative of the cross-entropy loss $l(\mathbf{y},\hat{\mathbf{y}})$ for softmax.
    1. Compute the variance of the distribution given by $\mathrm{softmax}(\mathbf{o})$ and show that it matches the second derivative computed above.
1. Assume that we have three classes which occur with equal probability, i.e., the probability vector is $(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$.
    1. What is the problem if we try to design a binary code for it?
    1. Can you design a better code? Hint: what happens if we try to encode two independent observations? What if we encode $n$ observations jointly?
1. When encoding signals transmitted over a physical wire, engineers do not always use binary codes. For instance, [PAM-3](https://en.wikipedia.org/wiki/Ternary_signal) uses three signal levels $\{-1, 0, 1\}$ as opposed to two levels $\{0, 1\}$. How many ternary units do you need to transmit an integer in the range $\{0, \ldots, 7\}$? Why might this be a better idea in terms of electronics?
1. The [Bradley--Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) uses
a logistic model to capture preferences. For a user to choose between apples and oranges one
assumes scores $o_{\textrm{apple}}$ and $o_{\textrm{orange}}$. Our requirements are that larger scores should lead to a higher likelihood in choosing the associated item and that
the item with the largest score is the most likely one to be chosen :cite:`Bradley.Terry.1952`.
    1. Prove that softmax satisfies this requirement.
    1. What happens if you want to allow for a default option of choosing neither apples nor oranges? Hint: now the user has three choices.
1. Softmax gets its name from the following mapping: $\textrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$.
    1. Prove that $\textrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$.
    1. How small can you make the difference between both functions? Hint: without loss of
    generality you can set $b = 0$ and $a \geq b$.
    1. Prove that this holds for $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b)$, provided that $\lambda > 0$.
    1. Show that for $\lambda \to \infty$ we have $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$.
    1. Construct an analogous softmin function.
    1. Extend this to more than two numbers.
1. The function $g(\mathbf{x}) \stackrel{\textrm{def}}{=} \log \sum_i \exp x_i$ is sometimes also referred to as the [log-partition function](https://en.wikipedia.org/wiki/Partition_function_(mathematics)).
    1. Prove that the function is convex. Hint: to do so, use the fact that the first derivative amounts to the probabilities from the softmax function and show that the second derivative is the variance.
    1. Show that $g$ is translation invariant, i.e., $g(\mathbf{x} + b) = g(\mathbf{x})$.
    1. What happens if some of the coordinates $x_i$ are very large? What happens if they're all very small?
    1. Show that if we choose $b = \mathrm{max}_i x_i$ we end up with a numerically stable implementation.
1. Assume that we have some probability distribution $P$. Suppose we pick another distribution $Q$ with $Q(i) \propto P(i)^\alpha$ for $\alpha > 0$.
    1. Which choice of $\alpha$ corresponds to doubling the temperature? Which choice corresponds to halving it?
    1. What happens if we let the temperature approach $0$?
    1. What happens if we let the temperature approach $\infty$?